# Tambahan — Klasifikasi Biner: lima hal yang belum ada di `cheatsheet-biner.ipynb`

File ini **bukan pengganti** `cheatsheet-biner.ipynb`, melainkan penambal. Tiap bagian berdiri
sendiri: baca judulnya, kalau situasinya cocok, salin blok kodenya ke notebook utama.

| § | Lubang yang ditambal | Kapan kamu butuh ini |
|---|---|---|
| 1 | Tokenizer membuang emoji & karakter non-ASCII | data media sosial, ulasan, komentar |
| 2 | Tidak ada cross-validation | diminta "lakukan 5-fold CV" |
| 3 | Tidak ada tuning hyperparameter | diminta "cari parameter terbaik" |
| 4 | Hanya menerima satu kolom teks | data berupa **pasangan kalimat**, atau judul + isi |
| 5 | Hanya menerima satu file | dosen memberi `train.csv` + `test.csv`, atau test tanpa label |

Setiap bagian mencetak **perbandingan angka sebelum vs sesudah**, jadi kelihatan apakah
tambalannya benar-benar berguna untuk datamu atau tidak.

In [1]:
import re, json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

SEED = 42
print("siap")

siap


---
## §1 · Tokenizer yang tidak membuang emoji dan huruf beraksen

### Masalahnya

Semua notebook memakai `re.findall(r"[a-zA-Z]+", teks)`. Regex itu hanya menerima **huruf Latin
polos**, jadi apa pun di luar itu hilang tanpa peringatan:

In [2]:
contoh = ["barangnya bagus bgt 😍 pengiriman cepet 🚀",
          "barang datang rusak 😡 seller susah dihubungi",
          "covid19 melonjak lagi",
          "naïve café résumé",
          "产品很好"]

print(f"{'teks':45s} {'hasil regex [a-zA-Z]+'}")
for t in contoh:
    print(f"  {t:43s} {re.findall(r'[a-zA-Z]+', t.lower())}")

teks                                          hasil regex [a-zA-Z]+
  barangnya bagus bgt 😍 pengiriman cepet 🚀    ['barangnya', 'bagus', 'bgt', 'pengiriman', 'cepet']
  barang datang rusak 😡 seller susah dihubungi ['barang', 'datang', 'rusak', 'seller', 'susah', 'dihubungi']
  covid19 melonjak lagi                       ['covid', 'melonjak', 'lagi']
  naïve café résumé                           ['na', 've', 'caf', 'r', 'sum']
  产品很好                                        []


Perhatikan: **semua emoji hilang**. Untuk analisis sentimen itu masalah serius — 😍 dan 😡
adalah penanda kelas yang jauh lebih jelas daripada kata mana pun di kalimatnya.

### Tambalannya

Tiga perubahan pada fungsi `bersihkan`:

1. `[^\W\d_]+` — "huruf apa pun menurut Unicode": mencakup é, ï, ü, dan aksara non-Latin.
2. Emoji ditangkap terpisah lewat rentang kode Unicode, lalu **dijadikan token sendiri**.
3. Angka yang menempel pada kata (`covid19`) tidak lagi memotong katanya.

In [3]:
# rentang blok emoji utama di Unicode
EMOJI = re.compile(
    "[" "\U0001F300-\U0001FAFF"   # simbol & piktograf, emoji tambahan
        "\U00002600-\U000027BF"   # simbol lain-lain, dingbats
        "\U0001F1E6-\U0001F1FF"   # bendera
        "\U00002190-\U000021AA"   # panah
        "\U0000FE0F"              # variation selector
    "]")


def tokenize_unicode(teks, simpan_emoji=True):
    t = str(teks).lower()
    t = re.sub(r"http\S+|www\.\S+", " urltoken ", t)
    t = re.sub(r"@\w+", " usertoken ", t)

    emoji = EMOJI.findall(t) if simpan_emoji else []
    t = EMOJI.sub(" ", t)

    # [^\W\d_]+ = huruf menurut Unicode (bukan angka, bukan underscore, bukan tanda baca)
    kata = re.findall(r"[^\W\d_]+", t, re.UNICODE)
    return kata + emoji          # emoji ditaruh di belakang sebagai token tersendiri


print(f"{'teks':45s} hasil")
for t in contoh:
    print(f"  {t:43s} {tokenize_unicode(t)}")

teks                                          hasil
  barangnya bagus bgt 😍 pengiriman cepet 🚀    ['barangnya', 'bagus', 'bgt', 'pengiriman', 'cepet', '😍', '🚀']
  barang datang rusak 😡 seller susah dihubungi ['barang', 'datang', 'rusak', 'seller', 'susah', 'dihubungi', '😡']
  covid19 melonjak lagi                       ['covid', 'melonjak', 'lagi']
  naïve café résumé                           ['naïve', 'café', 'résumé']
  产品很好                                        ['产品很好']


### Apakah benar-benar berguna? Ukur, jangan diasumsikan

`data/ulasan_emoji.csv` berisi 40 ulasan Indonesia bergaya media sosial: slang, singkatan,
dan emoji. Kita bandingkan dua tokenizer pada data yang sama.

In [4]:
df_e = pd.read_csv("data/ulasan_emoji.csv")
print("jumlah dokumen:", len(df_e), "|", df_e["label"].value_counts().to_dict())
print("contoh:", df_e["text"].iloc[0])

tok_lama = lambda t: re.findall(r"[a-zA-Z]+", str(t).lower())

hasil = []
for nama, tok in [("lama  [a-zA-Z]+ (emoji dibuang)", tok_lama),
                  ("baru  unicode + emoji disimpan", tokenize_unicode)]:
    pipe = Pipeline([("tfidf", TfidfVectorizer(tokenizer=tok, lowercase=False,
                                               token_pattern=None, ngram_range=(1, 2), min_df=1)),
                     ("clf", LogisticRegression(max_iter=1000))])
    cv = cross_val_score(pipe, df_e["text"], df_e["label"],
                         cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                         scoring="f1_macro")
    n_fitur = len(pipe.fit(df_e["text"], df_e["label"]).named_steps["tfidf"].vocabulary_)
    hasil.append((nama, n_fitur, cv.mean(), cv.std()))

print()
print(pd.DataFrame(hasil, columns=["tokenizer", "jumlah fitur", "cv f1_macro", "std"])
        .round(3).to_string(index=False))

jumlah dokumen: 40 | {'positif': 20, 'negatif': 20}
contoh: aplikasinya enteng, gk pernah error 👌 fiturnya lengkap

                      tokenizer  jumlah fitur  cv f1_macro   std
lama  [a-zA-Z]+ (emoji dibuang)           402        0.657 0.176
 baru  unicode + emoji disimpan           492        0.677 0.144


In [5]:
# Emoji mana yang jadi penanda kelas? Ini bahan bagus untuk laporan.
pipe = Pipeline([("tfidf", TfidfVectorizer(tokenizer=tokenize_unicode, lowercase=False,
                                           token_pattern=None, min_df=1)),
                 ("clf", LogisticRegression(max_iter=1000))]).fit(df_e["text"], df_e["label"])
nama = np.array(pipe.named_steps["tfidf"].get_feature_names_out())
koef = pipe.named_steps["clf"].coef_[0]
kelas = pipe.named_steps["clf"].classes_

emoji_saja = [i for i, w in enumerate(nama) if EMOJI.fullmatch(w)]
if emoji_saja:
    s = pd.Series(koef[emoji_saja], index=nama[emoji_saja]).sort_values()
    print(f"emoji penanda '{kelas[0]}':", ", ".join(s.head(6).index))
    print(f"emoji penanda '{kelas[1]}':", ", ".join(s.tail(6).index[::-1]))
    print(f"\ntotal {len(emoji_saja)} emoji jadi fitur -- semuanya hilang di tokenizer lama")

emoji penanda 'negatif': 😡, 😐, 😤, 😒, 😞, 😑
emoji penanda 'positif': 😊, 😎, 👕, 🔋, 💡, 🙏

total 44 emoji jadi fitur -- semuanya hilang di tokenizer lama


### Cara membaca hasilnya — jangan langsung percaya

Perhatikan kolom `std`. Kenaikan f1 dari tokenizer baru hanya sekitar **+0,02**, sementara
simpangan bakunya **0,14–0,18**. Artinya: pada dataset 40 dokumen ini, perbedaan itu
**masih di dalam rentang derau** dan belum bisa disebut peningkatan nyata.

Yang bisa diklaim dengan jujur:

1. **44 emoji kini jadi fitur** yang sebelumnya hilang total — itu fakta, bukan tafsiran.
2. Emoji yang terpilih **masuk akal**: 😡 😤 😒 ke negatif, 😊 😎 🙏 ke positif.
3. Beberapa emoji lain (👕 🔋 💡) ikut terangkat padahal netral — itu jelas artefak data 40 baris.
4. Efeknya diperkirakan lebih besar pada data media sosial yang sungguhan (ribuan baris),
   karena emoji muncul cukup sering untuk menjadi sinyal yang stabil.

Untuk laporan, kalimat yang aman: *"tokenizer Unicode menyelamatkan 44 token emoji yang
sebelumnya dibuang; pada dataset kecil ini peningkatan f1 belum signifikan secara statistik."*

> **Cara menempelkan ke `cheatsheet-biner.ipynb`:** ganti isi `re.findall(r"[a-zA-Z]+", t)`
> di dalam fungsi `bersihkan` (§4) dengan pemanggilan `tokenize_unicode`. Kalau datanya tidak
> mengandung emoji sama sekali, tokenizer lama sudah cukup — tidak perlu diganti.

---
## §2 · Cross-validation

### Masalahnya

`cheatsheet-biner.ipynb` hanya memakai **satu** train/test split. Skornya bisa naik-turun
besar hanya karena kebetulan pembagian datanya. Pada dataset kecil ini efeknya besar sekali:

In [6]:
df = pd.read_csv("data/sms_spam.csv").rename(columns={"message": "text"})[["text", "label"]]

pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
                 ("clf", LogisticRegression(max_iter=1000))])

# satu split, diulang dengan random_state berbeda
skor = []
for rs in range(8):
    Xtr, Xte, ytr, yte = train_test_split(df["text"], df["label"], test_size=0.2,
                                          stratify=df["label"], random_state=rs)
    skor.append(f1_score(yte, pipe.fit(Xtr, ytr).predict(Xte), average="macro"))

print("f1 dari 8 split berbeda:", [round(float(s), 3) for s in skor])
print(f"  terendah {min(skor):.3f} | tertinggi {max(skor):.3f} | selisih {max(skor)-min(skor):.3f}")
print("\n-> angka dari SATU split tidak bisa dipercaya sendirian")

f1 dari 8 split berbeda: [0.746, 1.0, 1.0, 0.875, 0.873, 1.0, 1.0, 0.806]
  terendah 0.746 | tertinggi 1.000 | selisih 0.254

-> angka dari SATU split tidak bisa dipercaya sendirian


### Tambalannya

`cross_val_score` melatih model k kali pada pembagian berbeda, lalu melaporkan rata-rata
**dan simpangan bakunya**. Yang dilaporkan di laporan: `mean ± std`, bukan satu angka.

In [7]:
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)   # stratified: proporsi kelas terjaga
skor = cross_val_score(pipe, df["text"], df["label"], cv=cv, scoring="f1_macro", n_jobs=-1)

print("skor tiap fold:", skor.round(3).tolist())
print(f"rata-rata     : {skor.mean():.3f} +/- {skor.std():.3f}")

# bandingkan beberapa model sekaligus, semuanya pakai CV yang sama
baris = []
for nama, clf in [("MultinomialNB", MultinomialNB()),
                  ("LogisticRegression", LogisticRegression(max_iter=1000)),
                  ("LinearSVC", LinearSVC())]:
    p = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
                  ("clf", clf)])
    s = cross_val_score(p, df["text"], df["label"], cv=cv, scoring="f1_macro", n_jobs=-1)
    baris.append((nama, round(s.mean(), 3), round(s.std(), 3)))

print()
print(pd.DataFrame(baris, columns=["model", "cv f1_macro", "std"]).to_string(index=False))

skor tiap fold: [0.875, 0.937, 0.937, 0.806, 0.937]
rata-rata     : 0.898 +/- 0.052



             model  cv f1_macro   std
     MultinomialNB        0.899 0.064
LogisticRegression        0.898 0.052
         LinearSVC        0.887 0.074


> **Kenapa `Pipeline` wajib di sini:** `cross_val_score` memisahkan data **sebelum** memanggil
> `fit`. Kalau vectorizer di-`fit` di luar, nilai IDF-nya sudah dihitung dari seluruh data
> termasuk fold validasi → data leakage → skor CV-nya bohong.
>
> Perhatikan juga `std`. Kalau simpangannya besar (> 0,05 seperti di sini), perbedaan antar
> model yang kecil **tidak bermakna** — jangan mengklaim satu model lebih baik hanya karena
> selisih 0,01.

---
## §3 · Tuning hyperparameter

`GridSearchCV` mencoba semua kombinasi dan memilih yang terbaik **menurut cross-validation**,
bukan menurut test set. Format namanya: `<nama_step>__<nama_parameter>`.

In [8]:
grid = {
    "tfidf__ngram_range":  [(1, 1), (1, 2)],
    "tfidf__min_df":       [1, 2],
    "tfidf__sublinear_tf": [False, True],
    "clf__C":              [0.5, 1.0, 5.0],
}

gs = GridSearchCV(pipe, grid, cv=cv, scoring="f1_macro", n_jobs=-1)
gs.fit(df["text"], df["label"])

print("jumlah kombinasi dicoba:", len(gs.cv_results_["params"]))
print("parameter terbaik      :", gs.best_params_)
print("skor CV terbaik        :", round(gs.best_score_, 3))

print("\n5 kombinasi teratas:")
hasil = pd.DataFrame(gs.cv_results_)[["params", "mean_test_score", "std_test_score"]]
hasil = hasil.sort_values("mean_test_score", ascending=False).head(5)
for _, r in hasil.iterrows():
    ringkas = {k.split("__")[1]: v for k, v in r["params"].items()}
    print(f"  {r['mean_test_score']:.3f} +/- {r['std_test_score']:.3f}  {ringkas}")

model_terbaik = gs.best_estimator_

jumlah kombinasi dicoba: 24
parameter terbaik      : {'clf__C': 0.5, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1), 'tfidf__sublinear_tf': False}
skor CV terbaik        : 0.95

5 kombinasi teratas:
  0.950 +/- 0.025  {'C': 0.5, 'min_df': 1, 'ngram_range': (1, 1), 'sublinear_tf': False}
  0.950 +/- 0.025  {'C': 0.5, 'min_df': 1, 'ngram_range': (1, 1), 'sublinear_tf': True}
  0.950 +/- 0.025  {'C': 1.0, 'min_df': 1, 'ngram_range': (1, 1), 'sublinear_tf': True}
  0.950 +/- 0.025  {'C': 1.0, 'min_df': 1, 'ngram_range': (1, 1), 'sublinear_tf': False}
  0.950 +/- 0.025  {'C': 5.0, 'min_df': 1, 'ngram_range': (1, 1), 'sublinear_tf': False}


> **Jebakan:** jangan menjalankan `GridSearchCV` pada seluruh data lalu melaporkan
> `best_score_` sebagai hasil akhir — itu skor validasi, dan parameternya sudah "melihat" data
> tersebut. Alurnya: `GridSearchCV` di **train**, lalu ukur sekali di **test** yang tidak
> tersentuh.

---
## §4 · Data berupa DUA kolom teks

### Masalahnya

Semua config berasumsi satu `TEXT_COL`. Ada dua situasi yang tidak muat:

**(a) Judul + isi** — dua kolom yang sebenarnya satu dokumen. Ini mudah: gabung saja.

**(b) Pasangan kalimat** — dua kalimat yang **dibandingkan**: deteksi duplikat, parafrase,
entailment. Ini tidak bisa sekadar digabung, karena yang ditanyakan adalah **hubungan** antar
keduanya, bukan isi gabungannya.

In [9]:
# ---- kasus (a): judul + isi, cukup digabung ----
# df["text"] = df["judul"].fillna("") + " " + df["isi"].fillna("")

# ---- kasus (b): pasangan kalimat ----
dp = pd.read_csv("data/pasangan_kalimat.csv")
print("kolom:", list(dp.columns), "| baris:", len(dp))
print(dp["label"].value_counts().to_dict())
print()
print(dp.head(3).to_string(index=False))

kolom: ['kalimat1', 'kalimat2', 'label'] | baris: 40
{'duplikat': 20, 'bukan': 20}

                                kalimat1                           kalimat2    label
     Apakah produk ini bergaransi resmi? Barang ini garansinya resmi tidak? duplikat
Bagaimana cara reset password akun saya? Berapa lama pengiriman ke Bandung?    bukan
     Bagaimana cara membatalkan pesanan?  Apakah barang ini masih tersedia?    bukan


### Tambalan 1 — gabung dengan penanda pemisah (baseline)

Cara termudah: satukan jadi satu string dengan token pemisah. Modelnya tetap TF-IDF biasa.

In [10]:
Xg = dp["kalimat1"] + " sepsep " + dp["kalimat2"]     # "sepsep" = penanda batas, satu token
ytarget = dp["label"]

pipe_gabung = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
                        ("clf", LogisticRegression(max_iter=1000))])
s = cross_val_score(pipe_gabung, Xg, ytarget, cv=StratifiedKFold(5, shuffle=True,
                    random_state=SEED), scoring="f1_macro", n_jobs=-1)
print(f"gabung + TF-IDF   : f1_macro {s.mean():.3f} +/- {s.std():.3f}")

gabung + TF-IDF   : f1_macro 0.044 +/- 0.054


### Tambalan 2 — fitur kemiripan (lebih tepat untuk pasangan kalimat)

Cara di atas lemah karena TF-IDF tidak tahu kata mana milik kalimat 1 dan mana milik kalimat 2 —
padahal yang menentukan "duplikat" justru **seberapa banyak keduanya beririsan**. Jadi buat
fiturnya secara eksplisit:

| Fitur | Arti |
|---|---|
| cosine similarity TF-IDF | kemiripan bobot kata kedua kalimat |
| Jaccard | proporsi kata yang sama dari total kata unik |
| jumlah kata bersama | irisan mentah |
| selisih panjang | pasangan duplikat biasanya panjangnya mirip |

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
vec.fit(pd.concat([dp["kalimat1"], dp["kalimat2"]]))      # vocabulary dari kedua kolom
A, B = vec.transform(dp["kalimat1"]), vec.transform(dp["kalimat2"])

kata1 = dp["kalimat1"].map(lambda s: set(re.findall(r"[^\W\d_]+", s.lower())))
kata2 = dp["kalimat2"].map(lambda s: set(re.findall(r"[^\W\d_]+", s.lower())))

fitur = pd.DataFrame({
    "cosine":   [float(cosine_similarity(A[i], B[i])[0, 0]) for i in range(len(dp))],
    "jaccard":  [len(a & b) / max(len(a | b), 1) for a, b in zip(kata1, kata2)],
    "bersama":  [len(a & b) for a, b in zip(kata1, kata2)],
    "beda_len": (dp["kalimat1"].str.split().str.len()
                 - dp["kalimat2"].str.split().str.len()).abs(),
})
print(fitur.head(4).round(3).to_string(index=False))
print("\nrata-rata per kelas:")
print(fitur.groupby(ytarget.values).mean().round(3).to_string())

 cosine  jaccard  bersama  beda_len
  0.185     0.25        2         0
  0.000     0.00        0         1
  0.000     0.00        0         1
  0.000     0.00        0         1

rata-rata per kelas:
          cosine  jaccard  bersama  beda_len
bukan      0.010    0.019     0.15       1.1
duplikat   0.173    0.223     1.65       0.7


In [12]:
s2 = cross_val_score(LogisticRegression(max_iter=1000), fitur, ytarget,
                     cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                     scoring="f1_macro", n_jobs=-1)
print(f"gabung + TF-IDF        : f1_macro {s.mean():.3f} +/- {s.std():.3f}")
print(f"fitur kemiripan (4 kol): f1_macro {s2.mean():.3f} +/- {s2.std():.3f}")
print("\n-> untuk pasangan kalimat, 4 fitur kemiripan mengalahkan ribuan fitur TF-IDF,")
print("   karena yang ditanyakan memang HUBUNGAN antar kalimat, bukan isinya.")

gabung + TF-IDF        : f1_macro 0.044 +/- 0.054
fitur kemiripan (4 kol): f1_macro 0.820 +/- 0.065

-> untuk pasangan kalimat, 4 fitur kemiripan mengalahkan ribuan fitur TF-IDF,
   karena yang ditanyakan memang HUBUNGAN antar kalimat, bukan isinya.


---
## §5 · Dua file (`train.csv` + `test.csv`) dan test tanpa label

### Kalau dosen memberi dua file

Yang berubah cuma satu: **jangan panggil `train_test_split`**. Sisanya sama persis.

In [13]:
LABELS = {0: "negatif", 4: "positif"}


def baca(path):
    d = pd.read_csv(path)
    d = d.rename(columns={"target": "label"})[["text", "label"]]
    d["label"] = d["label"].map(LABELS).fillna(d["label"])
    return d.dropna().drop_duplicates(subset=["text"]).reset_index(drop=True)


train = baca("data/split/train.csv")
test  = baca("data/split/test.csv")
print("train:", len(train), "| test:", len(test))

# cek kebocoran -- WAJIB sebelum melatih apa pun
bocor = set(train["text"]) & set(test["text"])
print("dokumen bocor train<->test:", len(bocor))
if bocor:
    test = test[~test["text"].isin(bocor)].reset_index(drop=True)

print("label asing di test:", set(test["label"]) - set(train["label"]))

train: 2400 | test: 800
dokumen bocor train<->test: 0
label asing di test: set()


In [14]:
model = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
                  ("clf", LogisticRegression(max_iter=1000))])

# semua eksperimen memakai CV di TRAIN -- test disentuh sekali saja di akhir
skor_cv = cross_val_score(model, train["text"], train["label"],
                          cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                          scoring="f1_macro", n_jobs=-1)
print(f"CV di train : {skor_cv.mean():.3f} +/- {skor_cv.std():.3f}")

model.fit(train["text"], train["label"])
pred = model.predict(test["text"])
print(f"TEST (sekali): f1_macro {f1_score(test['label'], pred, average='macro'):.3f}\n")
print(classification_report(test["label"], pred, zero_division=0))

CV di train : 0.694 +/- 0.019


TEST (sekali): f1_macro 0.681



              precision    recall  f1-score   support

     negatif       0.69      0.66      0.67       400
     positif       0.67      0.71      0.69       400

    accuracy                           0.68       800
   macro avg       0.68      0.68      0.68       800
weighted avg       0.68      0.68      0.68       800



### Kalau `test.csv` tidak punya kolom label

Gaya kompetisi: kamu hanya bisa memprediksi lalu menyerahkan hasilnya. Validasi tetap dari train.

In [15]:
test_nl = pd.read_csv("data/split/test_unlabeled.csv")
print("kolom:", list(test_nl.columns), "| baris:", len(test_nl))

pred_nl = model.predict(test_nl["text"].astype(str))
submission = pd.DataFrame({"id": test_nl["id"], "label": pred_nl})
submission.to_csv("submission.csv", index=False)

print("\nsubmission.csv tersimpan:")
print(submission.head(5).to_string(index=False))
print("\ndistribusi prediksi:", submission["label"].value_counts().to_dict())

kolom: ['text', 'id'] | baris: 800

submission.csv tersimpan:
 id   label
  0 negatif
  1 positif
  2 positif
  3 positif
  4 positif

distribusi prediksi: {'positif': 421, 'negatif': 379}


---
## Ringkasan: blok mana ditempel ke mana

| Kalau situasinya… | Salin dari | Tempel ke `cheatsheet-biner.ipynb` |
|---|---|---|
| Data berisi emoji / aksen | §1 `tokenize_unicode` | ganti `re.findall` di fungsi `bersihkan` (§4) |
| Diminta cross-validation | §2 `cross_val_score` | tambahkan setelah §7 (setelah model dilatih) |
| Diminta tuning | §3 `GridSearchCV` | ganti `model.fit(...)` di §7 |
| Kolom judul + isi | §4 baris `df["text"] = ...` | tepat setelah §3 (setelah kolom diambil) |
| Pasangan kalimat | §4 tambalan 2 | menggantikan §5–§7 seluruhnya |
| Dua file train/test | §5 | menggantikan §5 (split) di notebook utama |
| Test tanpa label | §5 blok terakhir | tambahkan setelah §9 |

## Yang masih belum ditambal

Sengaja tidak dimasukkan karena kemungkinan besar tidak akan diminta di praktikum sesi ini:

- **Ordinal** (rating 1–5 yang urutannya bermakna) — diperlakukan multiclass biasa saja, itu lazim.
- **Data jutaan baris** — perlu `HashingVectorizer` + `SGDClassifier.partial_fit`; untuk praktikum
  cukup `SAMPLE` di config.
- **Hierarchical classification** dan **explainability (LIME/SHAP)** — jarang diminta di sesi awal.